# Ingestion — Source System Entry Point

Discovers and executes all active ingestion tasks for a given source system.

**Input:** `config_master_id` and `source_system_id`.
The notebook queries `config_master` to find the correct child config table
(e.g. `rdbms_ingestion_config`, `nosql_ingestion_config`, `s3_config_master`),
fetches all active rows (`Is_Active = 1`) for the resolved source name,
and runs them sequentially.

**All source types use the same flow** — RDBMS, NoSQL, and S3.
The factory routes to the right connector based on `config_source_system.source_type`.

**Fault tolerance:** a failure on one table does NOT stop the others.
All objects are attempted; a summary is printed at the end. The notebook
raises a final exception only if at least one table failed.

In [0]:
%pip install python-dotenv --quiet
dbutils.library.restartPython()

In [0]:
%pip install paramiko boto3 --quiet


In [0]:
import sys
from datetime import date, datetime
sys.path.append("..")

from concurrent.futures import ThreadPoolExecutor, as_completed
from ingestion.utils.config_manager import (
    AUDIT_STATUS_FAILED,
    AUDIT_STATUS_SUCCESS,
    AUDIT_STATUS_SKIPPED,
    ConfigManager,
    IngestionTaskConfig,
    SOURCE_SYSTEM_TABLE,
    CONFIG_MASTER_TABLE,
    AUDIT_TABLE,
    DEPENDENCY_TABLE,
)
from ingestion.utils.orchestrator import IngestionOrchestrator

### Widgets

In [0]:
dbutils.widgets.text("config_master_id",    "",               "Config Master ID (int — routes to correct child config table)")
dbutils.widgets.text("source_system_id",    "",               "Source System ID (int — fetches credentials + source_name)")
dbutils.widgets.text("target_catalog",      "",               "Target Catalog (e.g. main, hive_metastore)")
dbutils.widgets.text("pipeline_name",       "",               "Pipeline Name (required)")
dbutils.widgets.text("job_run_id",          "",               "Job Run ID (required) — set to {{job.run_id}} in job config")
# dbutils.widgets.text("trigger_type",        "",               "Trigger Type (required) — SCHEDULED | MANUAL | EVENT")
dbutils.widgets.text("landing_volume_path", "",               "Landing Volume Base Path (blank = skip landing write)")
dbutils.widgets.text("environment",         "dev",            "Environment: dev | uat | prod")
dbutils.widgets.text("audit_table",         AUDIT_TABLE,      "Audit Table (override)")
dbutils.widgets.text("batch_start_date",    "1",              "Batch Start Date")
dbutils.widgets.text("max_workers",         "4",              "Max parallel workers (lookup + extraction per task)")
dbutils.widgets.text("dependency_table",    DEPENDENCY_TABLE, "Dependency Table (override)")
dbutils.widgets.text("silver_notebook_path",    "",           "Workspace path to Silver transformation notebook (blank = skip Silver trigger)")
dbutils.widgets.text("silver_notebook_timeout", "3600",       "Max seconds to wait for each Silver notebook run")

In [0]:
process_timestamp = datetime.utcnow()
print(f"Using process_timestamp for this ingestion batch: {process_timestamp}")

config_master_id_raw = dbutils.widgets.get("config_master_id") or None
source_system_id_raw = dbutils.widgets.get("source_system_id") or None

if not config_master_id_raw or not source_system_id_raw:
    dbutils.notebook.exit("Error: config_master_id and source_system_id are required.")

config_master_id     = int(config_master_id_raw)
source_system_id     = int(source_system_id_raw)

target_catalog       = dbutils.widgets.get("target_catalog")       or "hive_metastore"
pipeline_name        = dbutils.widgets.get("pipeline_name")        or None
try:
    job_id              = dbutils.widgets.get("job_id")           or None
except Exception:
    job_id              = None
job_run_id           = dbutils.widgets.get("job_run_id")           or None
# trigger_type         = dbutils.widgets.get("trigger_type")         or None

if not pipeline_name:
    dbutils.notebook.exit("Error: pipeline_name widget is required and cannot be empty.")
if not job_run_id:
    dbutils.notebook.exit("Error: job_run_id widget is required and cannot be empty.")
# if not trigger_type:
#     dbutils.notebook.exit("Error: trigger_type widget is required and cannot be empty.")

landing_volume_path  = dbutils.widgets.get("landing_volume_path")  or None
environment          = dbutils.widgets.get("environment")          or "dev"
audit_table          = dbutils.widgets.get("audit_table")          or AUDIT_TABLE
batch_start_date_raw = dbutils.widgets.get("batch_start_date")     or "1"
max_workers          = int(dbutils.widgets.get("max_workers") or "4")

from ingestion.utils.logger import get_logger
logger = get_logger(environment=environment)

# Parse batch_start_date_raw to datetime object if it is from the orchestrator
from datetime import datetime
parsed_batch_start_date = None
if batch_start_date_raw and str(batch_start_date_raw).strip() != "1":
    try:
        clean_dt = str(batch_start_date_raw).replace("T", " ").split(".")[0]
        parsed_batch_start_date = datetime.strptime(clean_dt, "%Y-%m-%d %H:%M:%S")
        print(f"Using parsed batch_start_date: {parsed_batch_start_date}")
    except Exception as parse_err:
        print(f"[Warning] Failed to parse batch_start_date '{batch_start_date_raw}': {parse_err}")
dependency_table     = dbutils.widgets.get("dependency_table")     or DEPENDENCY_TABLE

silver_notebook_path    = dbutils.widgets.get("silver_notebook_path")    or None
silver_notebook_timeout = int(dbutils.widgets.get("silver_notebook_timeout") or "3600")


## Get Databricks Job Context

Job/run information comes from the Databricks runtime.
Nothing is hardcoded.

In [0]:
def get_databricks_job_context():

    context = (
        dbutils.notebook.entry_point
        .getDbutils()
        .notebook()
        .getContext()
    )

    def get_context_value(method_name):

        try:
            return getattr(context, method_name)().get()
        except Exception:
            return None
    databricks_url = get_context_value("apiUrl")
    try:
        job_id = dbutils.widgets.get("job_id")
    except Exception:
        job_id = None

    databricks_url = (
        f"{databricks_url}/#job/{job_id}"
        if databricks_url and job_id
        else None
    )
    return {
        "job_id": get_context_value("jobId"),
        "job_name": get_context_value("jobName"),
        "notebook_name": get_context_value("notebookPath"),
        "databricks_url": databricks_url,
        "trigger_type": get_context_value("triggerType"),
        "trigger_id": get_context_value("triggerId"),
        "trigger_name": get_context_value("triggerName"),
    }


job_context = get_databricks_job_context()

# ── job_run_id and trigger_type from widget values ──
print(f"job_run_id   : {job_run_id}")
# print(f"trigger_type : {trigger_type}")

job_context["job_run_id"]   = job_run_id
# job_context["trigger_type"] = trigger_type

# pipeline_start_time is job-level — captured ONCE here, before the table
# fan-out below, and threaded through job_context so every table's
# DependencyLogger row uses the same value (see orchestrator.run()).
pipeline_start_time = datetime.utcnow()
job_context["pipeline_start_time"] = pipeline_start_time
print(f"pipeline_start_time (job-level): {pipeline_start_time}")

### Resolve pipeline name
Always read from the widget value.

In [0]:
print(f"pipeline_name from widget: '{pipeline_name}'")

### Discover ingestion tasks for this source

When running as part of a Databricks Job, Task 0 (`get_tasks.py`) queries
the config tables and publishes the active tasks via `taskValues`.
This task reads them from `taskValues` to avoid duplicate config queries.
Falls back to a direct config query when running the notebook standalone
(interactive / manual run without Task 0).

In [0]:
import json
from ingestion.utils.config_manager import SourceSystemConfig, IngestionTaskConfig

payload_str = None
try:
    payload_str = dbutils.jobs.taskValues.get(
        taskKey   = "get_table_details",
        key       = "active_tasks_metadata",
        default   = None,
        debugValue = None,
    )
except Exception:
    pass  # taskValues not available in standalone mode

# config_mgr is needed regardless of mode — IngestionOrchestrator uses it
# later for Silver_Last_Sink_Date bookkeeping (see orchestrator.run()), even
# in Job mode where task discovery itself is skipped (tasks already came
# from taskValues).
config_mgr = ConfigManager(
    spark,
    source_system_table = SOURCE_SYSTEM_TABLE,
    config_master_table = CONFIG_MASTER_TABLE,
    target_catalog      = target_catalog,
)

if payload_str:
    # ── Job mode: deserialize what get_tasks.py published ──────────────────
    print("[Tasks] Reading active tasks from taskValues (get_table_details task).")
    payload    = json.loads(payload_str)
    source_sys = SourceSystemConfig.from_dict(payload["source_sys"])
    tasks      = [IngestionTaskConfig.from_dict(t) for t in payload["tasks"]]
else:
    # ── Standalone mode: query config tables directly ──────────────────────
    print("[Tasks] taskValues not available — querying config tables directly (standalone mode).")
    source_sys, tasks = config_mgr.get_active_tasks(
        config_master_id = config_master_id,
        source_system_id = source_system_id,
        pipeline_name    = pipeline_name,
        batch_start_date = batch_start_date_raw,
    )

print(f"Resolved source : {source_sys.source_name} ({source_sys.source_type})")
print(f"Active tasks    : {len(tasks)}")

# Configure S3/Volume logging dynamically
resolved_landing_path = landing_volume_path or source_sys.landing_volume_path
if resolved_landing_path:
    s3_log_path = f"{resolved_landing_path.rstrip('/')}/logs/{pipeline_name}_{job_run_id}.log"
    from ingestion.utils.logger import configure_s3_logging
    configure_s3_logging(s3_log_path, dbutils=dbutils)

logger.info(f"Pipeline started for source: {source_sys.source_name} ({source_sys.source_type})")

if not tasks:
    dbutils.notebook.exit("No active ingestion tasks found for this pipeline.")

# NOTE: max_workers is now derived dynamically from the distinct batch_id count
# for this pipeline in the execution section below. The widget / config
# Max_Workers value is no longer used to size the thread pool.

### Execute Ingestion

In [0]:
# trigger_id also set to rootRunId for traceability in audit trigger_id column
trigger_id = job_run_id
job_context["trigger_id"] = trigger_id

job_context["trigger_id"] = trigger_id

orchestrator = IngestionOrchestrator(
    spark,
    dbutils,
    audit_table             = audit_table,
    dependency_table        = dependency_table,
    pipeline_name           = pipeline_name,
    environment             = environment,
    silver_notebook_path    = silver_notebook_path,
    silver_notebook_timeout = silver_notebook_timeout,
    config_mgr              = config_mgr,
)

import threading

def run_one(task: IngestionTaskConfig) -> dict:
    logger.info(f"Processing table {task.source_object_name}")
    """
    Run a single ingestion task — works for RDBMS, NoSQL, and S3.

    Retries happen inside IngestionOrchestrator.run(), scoped only to the
    source connection pull (connector.extract), using the source system's
    retry_count/retry_interval from config_source_system. Writing/transform
    steps are not retried — a failure there fails the task outright.
    """

    return orchestrator.run(
        source_sys          = source_sys,
        task                = task,
        config_master_id    = config_master_id,   # ← routing table ID from widget
        landing_volume_path = landing_volume_path,
        trigger_id          = trigger_id,
        # trigger_type        = trigger_type,
        job_context          = job_context,
        sink_batch_started_date = parsed_batch_start_date,
        process_timestamp     = process_timestamp,
    )



results = []

# ── Batch-level parallelism ────────────────────────────────────────────────
# batch_id  → controls PARALLEL execution: one thread per distinct batch.
# priority  → controls SEQUENTIAL execution of tables WITHIN a batch (ascending).
#
# max_workers is derived dynamically from the distinct batch_id count for this
# pipeline — NOT hardcoded, NOT taken from the widget/config. Tables are never
# assigned to their own threads; a batch's tables run one-by-one inside the
# batch's single thread. Per-table processing (load type, incremental/full,
# retry, timeout, watermark, …) is unchanged — it all still happens in run_one.

# Group tasks by batch_id, tables inside each batch ordered by priority ascending.
batches = {}
for task in sorted(tasks, key=lambda t: t.priority):
    batches.setdefault(task.batch_id, []).append(task)

max_workers = len(batches)   # distinct batch_id count for this pipeline


def run_batch(batch_id, batch_tasks: list) -> list:
    """Run every table in one batch sequentially, in priority order."""
    batch_results = []
    print(
        f"[Batch {batch_id}] Starting {len(batch_tasks)} table(s) sequentially: "
        f"{[t.source_object_name for t in batch_tasks]}"
    )
    for task in batch_tasks:
        try:
            batch_results.append(run_one(task))
        except Exception as exc:
            print(f"Task {task.source_object_name} (Config ID: {task.config_id}) failed with exception: {exc}")
            batch_results.append({
                "config_id": task.config_id,
                "run_id":   None,
                "status":   AUDIT_STATUS_FAILED,
                "rows_read": 0,
                "error":    str(exc),
            })
    return batch_results


print(
    f"\nStarting {len(tasks)} tasks across {len(batches)} batch(es) with "
    f"ThreadPoolExecutor (max_workers={max_workers})..."
)
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_batch = {
        executor.submit(run_batch, batch_id, batch_tasks): batch_id
        for batch_id, batch_tasks in batches.items()
    }

    for future in as_completed(future_to_batch):
        batch_id = future_to_batch[future]
        try:
            results.extend(future.result())
        except Exception as exc:
            print(f"Batch {batch_id} failed with exception: {exc}")
            for task in batches[batch_id]:
                results.append({
                    "config_id": task.config_id,
                    "run_id":   None,
                    "status":   AUDIT_STATUS_FAILED,
                    "rows_read": 0,
                    "error":    str(exc),
                })

In [0]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── 1. Setup API Authentication & Environment Variables ────────────────────
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host_url = ctx.apiUrl().get()
api_token = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {api_token}", "Content-Type": "application/json"}

INSTANCE_POOL_ID = "your-instance-pool-id" 
CHILD_NOTEBOOK_PATH = "/Workspace/Path/To/Your/Batch_Executor_Notebook"
SPARK_VERSION = "13.3.x-scala2.12" 

# We keep the local orchestrator instance ONLY so we can call 
# orchestrator.dependency.complete_job(job_run_id) at the very end.
orchestrator = IngestionOrchestrator(
    spark, dbutils, audit_table=audit_table, dependency_table=dependency_table,
    pipeline_name=pipeline_name, environment=environment, config_mgr=config_mgr
)
trigger_id = job_run_id

# ── 2. Group Tasks by Batch ID & Sort by Priority ──────────────────────────
batches = {}
for task in sorted(tasks, key=lambda t: t.priority):
    batches.setdefault(task.batch_id, []).append(task)

max_batch_workers = len(batches) # e.g., 5 batches = 5 parallel jobs

# ── 3. The API Dispatcher Function ─────────────────────────────────────────
def trigger_batch_job(batch_id: int, batch_tasks: list) -> dict:
    logger.info(f"Dispatching Job Compute for Batch {batch_id} with {len(batch_tasks)} tables")
    
    # Extract just the config_ids to pass to the child notebook
    config_ids_str = ",".join([str(t.config_id) for t in batch_tasks])
    
    payload = {
        "run_name": f"Batch_{batch_id}_Extraction",
        "new_cluster": {
            "instance_pool_id": INSTANCE_POOL_ID,
            "spark_version": SPARK_VERSION,
            "num_workers": 0,  # Single Node Cluster per batch
            "custom_tags": {"ResourceClass": "SingleNode"}
        },
        "notebook_task": {
            "notebook_path": CHILD_NOTEBOOK_PATH,
            "base_parameters": {
                "batch_config_ids": config_ids_str, # Passed as a comma-separated string
                "config_master_id": str(config_master_id),
                "source_system_id": str(source_system_id),
                "pipeline_name": pipeline_name or "",
                "job_run_id": str(trigger_id),
                "landing_volume_path": landing_volume_path or "",
                "environment": environment,
                "batch_start_date": str(batch_start_date_raw) or "1",
                "silver_notebook_path": silver_notebook_path or "",
                "silver_notebook_timeout": str(silver_notebook_timeout),
                "target_catalog": target_catalog
            }
        }
    }
    
    # Submit the Job
    submit_endpoint = f"{host_url}/api/2.1/jobs/runs/submit"
    submit_resp = requests.post(submit_endpoint, headers=headers, json=payload)
    submit_resp.raise_for_status()
    
    run_id = submit_resp.json().get("run_id")
    logger.info(f"Started Run ID {run_id} for Batch {batch_id}")
    
    # Poll for Completion
    get_endpoint = f"{host_url}/api/2.1/jobs/runs/get"
    while True:
        status_resp = requests.get(get_endpoint, headers=headers, params={"run_id": run_id})
        status_data = status_resp.json()
        state = status_data.get("state", {})
        
        if state.get("life_cycle_state") in ["TERMINATED", "SKIPPED", "INTERNAL_ERROR"]:
            if state.get("result_state") == "SUCCESS":
                return {"batch_id": batch_id, "run_id": run_id, "status": AUDIT_STATUS_SUCCESS}
            else:
                error_msg = state.get("state_message", "Unknown Error")
                raise Exception(f"Batch {batch_id} failed: {error_msg}")
        
        time.sleep(15)

# ── 4. Parallel Batch Execution ────────────────────────────────────────────
results = []
print(f"\nStarting {len(tasks)} tasks across {max_batch_workers} parallel Batch Jobs...")

with ThreadPoolExecutor(max_workers=max_batch_workers) as executor:
    future_to_batch = {
        executor.submit(trigger_batch_job, b_id, b_tasks): b_id 
        for b_id, b_tasks in batches.items()
    }
    
    for future in as_completed(future_to_batch):
        batch_id = future_to_batch[future]
        try:
            results.append(future.result())
        except Exception as exc:
            print(f"Batch {batch_id} failed with exception: {exc}")
            results.append({"batch_id": batch_id, "status": AUDIT_STATUS_FAILED, "error": str(exc)})

# Close dependency job
orchestrator.dependency.complete_job(job_run_id)

In [0]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── 1. Setup API Authentication & Environment Variables ────────────────────
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host_url = ctx.apiUrl().get()
api_token = ctx.apiToken().get()
headers = {"Authorization": f"Bearer {api_token}", "Content-Type": "application/json"}

INSTANCE_POOL_ID = "your-instance-pool-id" 
CHILD_NOTEBOOK_PATH = "/Workspace/Users/ashish.ujawane@ganitinc.com/pfl-ingestion-framework-dynamic-worker/databricks-ingestion-framework/src/main/Batch_Executor_Notebook"
SPARK_VERSION = "13.3.x-scala2.12" 
AUDIT_STATUS_SUCCESS = "SUCCESS"
AUDIT_STATUS_FAILED = "FAILED"

# Keep the local orchestrator instance ONLY to call complete_job at the end.
orchestrator = IngestionOrchestrator(
    spark, dbutils, audit_table=audit_table, dependency_table=dependency_table,
    pipeline_name=pipeline_name, environment=environment, config_mgr=config_mgr
)
trigger_id = job_run_id

# ── 2. Group Tasks by Batch ID & Sort by Priority ──────────────────────────
batches = {}
for task in sorted(tasks, key=lambda t: t.priority):
    batches.setdefault(task.batch_id, []).append(task)

max_batch_workers = len(batches)

# ── 3. The API Dispatcher Function ─────────────────────────────────────────
def trigger_batch_job(batch_id: int, batch_tasks: list) -> dict:
    logger.info(f"Dispatching Job Compute for Batch {batch_id} with {len(batch_tasks)} tables")
    
    # Extract just the config_ids to pass to the child notebook
    config_ids_str = ",".join([str(t.config_id) for t in batch_tasks])
    
    payload = {
        "run_name": f"Batch_{batch_id}_Extraction",
        "new_cluster": {
            "instance_pool_id": INSTANCE_POOL_ID,
            "spark_version": SPARK_VERSION,
            # ── CHANGED: Replaced static num_workers with autoscale ──
            "autoscale": {
                "min_workers": 1, # Drops to 1 node when only a few tables are left
                "max_workers": 8  # Max capacity for the initial 8-table rush
            },
            "spark_conf": {
                "spark.scheduler.mode": "FAIR" # Crucial: Allows concurrent Spark execution
            }
        },
        "notebook_task": {
            "notebook_path": CHILD_NOTEBOOK_PATH,
            "base_parameters": {
                "batch_config_ids": config_ids_str,
                "config_master_id": str(config_master_id),
                "source_system_id": str(source_system_id),
                "pipeline_name": pipeline_name or "",
                "job_run_id": str(trigger_id),
                "landing_volume_path": landing_volume_path or "",
                "environment": environment,
                "batch_start_date": str(batch_start_date_raw) or "1",
                "silver_notebook_path": silver_notebook_path or "",
                "silver_notebook_timeout": str(silver_notebook_timeout),
                "target_catalog": target_catalog
            }
        }
    }
    
    # Submit the Job
    submit_endpoint = f"{host_url}/api/2.1/jobs/runs/submit"
    submit_resp = requests.post(submit_endpoint, headers=headers, json=payload)
    submit_resp.raise_for_status()
    
    run_id = submit_resp.json().get("run_id")
    logger.info(f"Started Run ID {run_id} for Batch {batch_id}")
    
    # Poll for Completion
    get_endpoint = f"{host_url}/api/2.1/jobs/runs/get"
    while True:
        status_resp = requests.get(get_endpoint, headers=headers, params={"run_id": run_id})
        status_data = status_resp.json()
        state = status_data.get("state", {})
        
        if state.get("life_cycle_state") in ["TERMINATED", "SKIPPED", "INTERNAL_ERROR"]:
            if state.get("result_state") == "SUCCESS":
                return {"batch_id": batch_id, "run_id": run_id, "status": AUDIT_STATUS_SUCCESS}
            else:
                error_msg = state.get("state_message", "Unknown Error")
                raise Exception(f"Batch {batch_id} failed: {error_msg}")
        
        time.sleep(15)

# ── 4. Parallel Batch Execution ────────────────────────────────────────────
results = []
print(f"\nStarting {len(tasks)} tasks across {max_batch_workers} parallel Batch Jobs...")

with ThreadPoolExecutor(max_workers=max_batch_workers) as executor:
    future_to_batch = {
        executor.submit(trigger_batch_job, b_id, b_tasks): b_id 
        for b_id, b_tasks in batches.items()
    }
    
    for future in as_completed(future_to_batch):
        batch_id = future_to_batch[future]
        try:
            results.append(future.result())
        except Exception as exc:
            print(f"Batch {batch_id} failed with exception: {exc}")
            results.append({"batch_id": batch_id, "status": AUDIT_STATUS_FAILED, "error": str(exc)})

# Close dependency job
orchestrator.dependency.complete_job(job_run_id)

### Close the dependency job

pipeline_end_time isn't known until every table has finished — bulk-stamp
it (and the derived dependency_resolve_time) onto every dependency_master_config
row for this job_run_id in one shot, now that the fan-out above is done.

In [0]:
orchestrator.dependency.complete_job(job_run_id)

### Results summary

Silver now runs coupled — inline, synchronously — inside each table's
own orchestrator.run() call (see IngestionOrchestrator._trigger_silver),
right after that table's landing write and before its Bronze Delta
write. By the time a task's future resolves above, its Silver run (if
enabled) has already finished, so results already carry it under
"silver_result" — no separate wait step needed.

In [0]:
STATUS_ICONS = {
    AUDIT_STATUS_SUCCESS: "✅",
    AUDIT_STATUS_SKIPPED: "⏭️",
}

print(f"\n{'='*75}")
print(f"{'CONF ID':>8}  {'STATUS':<10}  {'ROWS':>8}  ERROR")
print(f"{'='*75}")
for r in sorted(results, key=lambda x: x["config_id"]):
    icon   = STATUS_ICONS.get(r["status"], "❌")
    error  = (r.get("error") or "")[:50]
    print(f"{r['config_id']:>8}  {icon} {r['status']:<8}  {r.get('rows_read', 0):>8}  {error}")
print(f"{'='*75}")

succeeded = [r for r in results if r["status"] == AUDIT_STATUS_SUCCESS]
skipped   = [r for r in results if r["status"] == AUDIT_STATUS_SKIPPED]
failed    = [r for r in results if r["status"] == AUDIT_STATUS_FAILED]
print(
    f"Total: {len(results)} | ✅ Succeeded: {len(succeeded)} | "
    f"⏭️ Skipped (0 rows): {len(skipped)} | ❌ Failed: {len(failed)}\n"
)

silver_results = [r["silver_result"] for r in results if r.get("silver_result")]

if silver_results:
    print(f"{'='*75}")
    print(f"{'CONF ID':>8}  {'SILVER STATUS':<14}  TARGET")
    print(f"{'='*75}")
    for r in sorted(silver_results, key=lambda x: x["config_id"]):
        icon = "✅" if r["status"] == "SUCCESS" else "❌"
        print(f"{r['config_id']:>8}  {icon} {r['status']:<12}  {r.get('target', '')}")
    print(f"{'='*75}")

silver_failed = [r for r in silver_results if r["status"] == "FAILED"]
print(
    f"Silver — Total: {len(silver_results)} | "
    f"✅ Succeeded: {len(silver_results) - len(silver_failed)} | ❌ Failed: {len(silver_failed)}\n"
)

In [0]:
from ingestion.utils.logger import _upload_on_exit

if failed:
    failed_ids = [r["config_id"] for r in failed]
    silver_failed_ids = [r["config_id"] for r in silver_failed]
    logger.critical(
        f"Pipeline cannot continue — {len(failed)} of {len(results)} ingestion object(s) FAILED. "
        f"Failed Config IDs: {failed_ids}"
        f"{len(silver_failed)} of {len(silver_results)} Silver trigger(s) FAILED "
        f"(Config IDs: {silver_failed_ids}). "
        f"Check the audit table and logs above for details."       
    )
    _upload_on_exit()


_upload_on_exit()
dbutils.notebook.exit(
    f"SUCCESS: {len(succeeded)}/{len(results)} objects ingested "
    f"({len(skipped)} skipped — 0 rows in source)."
)